In [ ]:
import os, importlib

REPO_URL = 'https://github.com/litcorp0/checkmaize.git'  # change if you fork the project

def repo_root():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            os.system(f'cd {repo} && git clean -fdq data/manifests && git checkout -q -- data/manifests')
            os.system(f'cd {repo} && git pull')
            return repo
        print('Repo not on this runtime yet. Cloning from GitHub...')
        result = os.system(f'git clone {REPO_URL} /content/checkmaize')
        if result != 0 or not os.path.exists(repo):
            print('Automatic clone failed. Likely causes:')
            print('  - the GitHub repo is private (make it public first), or')
            print('  - no internet on this runtime.')
            print('Manual fix - run this in a NEW cell, then re-run this cell:')
            print(f'  !git clone {REPO_URL} /content/checkmaize')
            print('Or drag the checkmaize folder into the Colab file explorer (into /content).')
            raise SystemExit
        return repo
    return os.path.abspath('..')

REPO = repo_root()
os.chdir(REPO)
print('Working in:', os.getcwd())

missing = []
for mod in ['numpy', 'PIL', 'pandas', 'yaml', 'sklearn', 'matplotlib', 'huggingface_hub', 'pytest']:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
if missing:
    print('installing missing packages:', missing)
    !pip install -q -r requirements.txt
    print('dependencies installed')
else:
    print('dependencies OK')


Working in: /content/checkmaize
dependencies OK


In [1]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())

import shutil, zipfile
from huggingface_hub import hf_hub_download

REPO_ID = 'mohanty/PlantVillage'

dst = 'data/raw/plantvillage'
shutil.rmtree(dst, ignore_errors=True)
os.makedirs(dst, exist_ok=True)

CLASS_MAP = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_': 'Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight': 'Northern_Leaf_Blight',
    'Corn_(maize)___healthy': 'healthy',
}

print('downloading data.zip (~2 GB, 10-30 minutes)...')
data_zip = hf_hub_download(REPO_ID, 'data.zip', repo_type='dataset')
print('downloaded:', data_zip)

train_txt = hf_hub_download(REPO_ID, 'splits/color_train.txt', repo_type='dataset')
test_txt = hf_hub_download(REPO_ID, 'splits/color_test.txt', repo_type='dataset')

extract_dir = '/content/pv_extract'
shutil.rmtree(extract_dir, ignore_errors=True)
with zipfile.ZipFile(data_zip) as z:
    z.extractall(extract_dir)

raw_root = None
for root, dirs, files in os.walk(extract_dir):
    if os.path.basename(root) == 'raw' and 'color' in dirs:
        raw_root = root
        break
if raw_root is None:
    raise SystemExit('could not find raw/color inside data.zip - unexpected layout')

def leaf_id_for(file_name):
    ident = file_name.split('___')[-1]
    ident = ident.split('copy')[0]
    for ext in ('.jpg', '.JPG', '.jpeg', '.png', '.PNG'):
        ident = ident.replace(ext, '')
    return ident.strip()

copied = 0
missing = 0
for txt in [train_txt, test_txt]:
    with open(txt) as f:
        for line in f:
            rel = line.strip()
            if not rel:
                continue
            parts = rel.split('/')
            if len(parts) < 4 or parts[0] != 'raw' or parts[1] != 'color':
                continue
            class_dir = parts[2]
            if class_dir not in CLASS_MAP:
                continue
            file_name = parts[3]
            src = os.path.join(raw_root, *parts[1:])
            if not os.path.exists(src):
                missing += 1
                continue
            leaf_id = leaf_id_for(file_name)
            out_dir = os.path.join(dst, CLASS_MAP[class_dir])
            os.makedirs(out_dir, exist_ok=True)
            shutil.copyfile(src, os.path.join(out_dir, f'{leaf_id}__{file_name}'))
            copied += 1

print(f'plantvillage extraction done: {copied} maize images copied ({missing} missing)')


downloading data.zip (~2 GB, 10-30 minutes)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


downloaded: /root/.cache/huggingface/hub/datasets--mohanty--PlantVillage/snapshots/9e97599868962bd0079b8db4b7f1efa9185fa1e7/data.zip
plantvillage extraction done: 3852 maize images copied (0 missing)


In [3]:
import os
os.chdir('/content')

# (If you are in BROWSER Colab and already uploaded a dataset zip to /content
#  yourself, this cell will find and use it automatically - nothing to do here.)

# ---- Download the Ghana dataset directly from Kaggle ----
# NOTE: your credentials are filled in below. DO NOT commit/push this notebook
# while they are here - tell your assistant when done so they can scrub them.
import os
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY'
os.environ['KAGGLE_API_TOKEN'] = 'YOUR_KAGGLE_KEY'
!pip install -q kagglehub
import kagglehub
path = kagglehub.dataset_download('nirmalsankalana/crop-pest-and-disease-detection')
print('downloaded and extracted to:', path)

import os, zipfile, shutil, glob
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())

NEEDED = {'Leaf blight': 'Leaf blight', 'Leaf spot': 'Leaf spot', 'Healthy': 'Healthy'}
NEEDED_LOWER = {k.lower(): k for k in NEEDED}

candidates = []
for name in ['archive.zip', 'Raw Data.zip', 'crop-pest-and-disease-detection.zip']:
    p = f'/content/{name}'
    if os.path.exists(p):
        candidates.append(p)
if not candidates:
    candidates = sorted(z for z in glob.glob('/content/*.zip') if 'checkmaize' not in z)

kaggle_dirs = sorted(glob.glob('/root/.cache/kagglehub/datasets/nirmalsankalana/crop-pest-and-disease-detection/versions/*'))
# Newer kagglehub versions use a Colab cache (e.g. /kaggle/input/...); the
# `path` returned by dataset_download is the real location - check it first.
kaggle_locs = [path] + [d for d in kaggle_dirs if d != path]

extract_dir = '/content/ccmt_extract'
shutil.rmtree(extract_dir, ignore_errors=True)

if candidates:
    zip_path = candidates[0]
    print('Using:', os.path.basename(zip_path))
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
elif kaggle_locs:
    loc = kaggle_locs[0]
    if os.path.isfile(loc) and loc.lower().endswith('.zip'):
        print('Using kagglehub zip:', loc)
        with zipfile.ZipFile(loc) as z:
            z.extractall(extract_dir)
    else:
        extract_dir = loc
        print('Using kagglehub download:', extract_dir)
else:
    print('No Ghana dataset found on the cloud machine yet.')
    print('The Kaggle download reported a location but no files were found.')
    print('Re-run this cell; if it keeps failing, tell your assistant.')
    raise SystemExit

dst = 'data/raw/ccmt_ghana'
shutil.rmtree(dst, ignore_errors=True)
os.makedirs(dst, exist_ok=True)
copied = {}

def copy_dir(src, key):
    target = os.path.join(dst, NEEDED[key])
    shutil.copytree(src, target)
    copied[key] = len(os.listdir(target))

nested_maize = None
for root, dirs, files in os.walk(extract_dir):
    if 'Maize' in dirs:
        nested_maize = os.path.join(root, 'Maize')
        break

if nested_maize and any(k.lower() in NEEDED_LOWER for k in os.listdir(nested_maize)):
    for sub in sorted(os.listdir(nested_maize)):
        if sub.lower() in NEEDED_LOWER:
            copy_dir(os.path.join(nested_maize, sub), NEEDED_LOWER[sub.lower()])
            print(sub, copied[NEEDED_LOWER[sub.lower()]])
else:
    for folder in sorted(os.listdir(extract_dir)):
        full = os.path.join(extract_dir, folder)
        if folder.startswith('Maize ') and os.path.isdir(full):
            key = folder[len('Maize '):].lower()
            if key in NEEDED_LOWER:
                copy_dir(full, NEEDED_LOWER[key])
                print(key, copied[NEEDED_LOWER[key]])

missing = [k for k in NEEDED if k not in copied]
assert not missing, f'Could not find classes {missing} in the dataset'
print('ccmt_ghana ready:', copied)


100%|██████████| 1.25G/1.25G [00:19<00:00, 67.3MB/s]

Extracting files...


downloaded and extracted to: /root/.cache/kagglehub/datasets/nirmalsankalana/crop-pest-and-disease-detection/versions/1
Using kagglehub download: /root/.cache/kagglehub/datasets/nirmalsankalana/crop-pest-and-disease-detection/versions/1
healthy 208
leaf blight 1006
leaf spot 1259
ccmt_ghana ready: {'Healthy': 208, 'Leaf blight': 1006, 'Leaf spot': 1259}


In [11]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
!python -m data.make_manifest
!python -m data.make_splits
!python -m pytest data/tests -v


wrote 6325 rows to data/manifests/raw.csv
  common_rust: 1192
  gray_leaf_spot: 1772
  healthy: 1370
  northern_leaf_blight: 1991
train: 5205 rows
val: 625 rows
test: 495 rows
domain_shift_train: 3852 rows
domain_shift_val: 377 rows
domain_shift_test: 2473 rows
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/checkmaize
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 3 items                                                              

data/tests/test_make_manifest.py::test_build_raw_manifest_maps_and_groups PASSED [ 33%]
data/tests/test_make_splits.py::test_splits_respect_leakage_and_axes PASSED [ 66%]
data/tests/test_make_splits.py::test_splits_deterministic PASSED         [100%]

============================== 3 passed in 0.04s ===============================


In [12]:
import os, shutil
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
shutil.make_archive('/content/splits', 'zip', 'data/manifests')
try:
    from google.colab import files
    files.download('/content/splits.zip')
    print('download started (browser Colab)')
except Exception:
    print('VS Code mode: the file is at /content/splits.zip')
    print('Drag it from the file explorer onto your computer, unzip it, and put the')
    print('CSV files into: checkmaize/data/manifests/')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

download started (browser Colab)


In [10]:
!mkdir -p /content/checkmaize/data/manifests